## Without web search

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

groq_model = ChatGroq(
    model="llama-3.3-70b-versatile"
)
agent = create_agent(model=groq_model)


In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [6]:
print(response['messages'][-1].content)

My training data is current up to December 2023, but I have access to more recent information via internet search.


## Add web search tool

In [8]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "tavily-python"], check=False)

CompletedProcess(args=['c:\\Users\\ramiz\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'tavily-python'], returncode=0)

In [9]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95324475,
   'raw_content': None},
  {'url': 'https://ballotpedia.org/Daniel_Lurie',
   'title': 'Daniel Lurie - Ballotpedia',
   'content': 'Daniel Lurie is the Mayor of San Francisco in California. He assumed office on January 8, 2025. His current term ends on January 8, 2029.',
   'score': 0.88799405,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly 

In [12]:
from langchain_groq import ChatGroq

# ✅ Use these models - better tool calling support on Groq
groq_model = ChatGroq(model="llama-3.1-8b-instant")  # try this first

# OR use Gemini which has excellent tool calling
from langchain_google_genai import ChatGoogleGenerativeAI
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

agent = create_agent(
    model=groq_model,  # Gemini is much better at tool calling
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")
response = agent.invoke({"messages": [question]})

In [13]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='47cc80b2-f847-4e57-a9e3-872c484d24ee'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '854r6nh9a', 'function': {'arguments': '{"query":"San Francisco current mayor"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 219, 'total_tokens': 237, 'completion_time': 0.034143936, 'completion_tokens_details': None, 'prompt_time': 0.058352184, 'prompt_tokens_details': None, 'queue_time': 0.056362865, 'total_time': 0.09249612}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfa8-d4bc-7011-b183-92670e30bee1-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'San Francisco current mayor'}, 'id': '854r6nh9a', 'type': 'tool_call'}], invalid

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r